# 07k — Blinded historical-imagery adjudication

This stage prepares Google Earth Pro review artifacts for the 9 Stage-07j near-miss OFFICE candidates, with the frozen 16 OFFICE candidates available as an optional reference set.

The visual review is intentionally **blinded** to user id, near-miss family, behavioral diagnostics, and BCL context. Historical imagery is used only to classify visible physical/land-use context; it does not identify an employer or automatically produce an OFFICE label.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import pandas as pd

REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH=os.environ.get('GEOLIFE_REPO_BRANCH','main')
REPO_DIR=Path(os.environ.get('GEOLIFE_REPO_DIR','/root/geolife'))

def run(cmd,cwd=None): subprocess.run(cmd,cwd=cwd,check=True)
if not (REPO_DIR/'.git').exists():
    run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)])
else:
    run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH])
    run(['git','-C',str(REPO_DIR),'checkout',BRANCH])
    run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'])

run([sys.executable,'-m','pip','install','-q','-e','.[dev]'],cwd=REPO_DIR)
for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path: sys.path.insert(0,p)

from geolife.model import HomeOfficeConfig, build_semantic_locations
spec=importlib.util.spec_from_file_location('s07k',REPO_DIR/'analysis'/'07k_historical_imagery_review.py')
assert spec is not None and spec.loader is not None
s07k=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=s07k
spec.loader.exec_module(s07k)
print('sha:',subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip())
print(s07k.synthetic_self_check())


In [ ]:
V=Path('/mnt/geolife-data')
C=V/'cache'/'cp2_v2'/'07k_historical_imagery_review'; C.mkdir(parents=True,exist_ok=True)

def find_stay_cache():
    candidates=[
        V/'cache'/'03a_user_behavior_deep_dive'/'stays_baseline_v1.pkl',
        V/'artifacts'/'03a'/'stays_baseline_v1.pkl',
        REPO_DIR/'artifacts'/'03a'/'stays_baseline_v1.pkl',
    ]
    for p in candidates:
        if p.exists(): return p
    matches=sorted(V.glob('**/stays_baseline_v1.pkl'))
    if matches: return matches[0]
    raise FileNotFoundError('Cannot find stays_baseline_v1.pkl')

STAYS=find_stay_cache()
AUDIT=V/'cache'/'cp2_v2'/'07j_near_miss_office_audit'/'near_miss_office_audit_private.pkl'
if not AUDIT.exists(): raise FileNotFoundError(f'Run Stage 07j first: {AUDIT}')

stays=pd.read_pickle(STAYS)
audit=pd.read_pickle(AUDIT)
semantic_stays,locations=build_semantic_locations(
    stays,
    config=HomeOfficeConfig(clustering_method='complete_link',location_max_diameter_m=200.0),
)
assert len(audit)==25
assert int(audit['audit_group'].eq('baseline').sum())==16
assert int(audit['audit_group'].isin(['margin_near','share_near']).sum())==9
print('07j audit reproduction: PASS (25 total / 9 near-miss)')
print('semantic users:',semantic_stays['user_id'].nunique(),'locations:',len(locations))


In [ ]:
spatial=s07k.build_spatial_review_panel(audit,locations,semantic_stays)
manifest=s07k.blinded_review_manifest(spatial)
key=s07k.unblinding_key(spatial)
priority=s07k.review_priority_ids(spatial)
rubric=s07k.review_rubric_table()

assert len(spatial)==25
assert len(priority)==9
assert manifest['audit_id'].nunique()==25
assert 'user_id' not in manifest.columns and 'audit_group' not in manifest.columns
assert spatial[['candidate_latitude','candidate_longitude','observation_median_date']].notna().all().all()
print('blinding + spatial/date validation: PASS')

print('MEDIAN OBSERVATION YEAR — ALL 25 (blinded aggregate)')
display(pd.to_datetime(spatial['observation_median_date']).dt.year.value_counts().sort_index().rename_axis('year').reset_index(name='locations'))
print('REVIEW RUBRIC')
display(rubric)


In [ ]:
# Private KML exports. The KML itself contains coordinates, so keep it private.
all_kml=s07k.write_review_kml(spatial,C/'all25_blinded_historical_imagery_review_private.kml')
near_ids=priority['audit_id'].tolist()
near_kml=s07k.write_review_kml(spatial,C/'near_miss_9_blinded_historical_imagery_review_private.kml',audit_ids=near_ids)
baseline_ids=spatial.loc[spatial['audit_group'].eq('baseline'),'audit_id'].tolist()
baseline_kml=s07k.write_review_kml(spatial,C/'baseline16_blinded_historical_imagery_reference_private.kml',audit_ids=baseline_ids)

# Blinded review sheets.
manifest.to_csv(C/'all25_blinded_review_manifest_private.csv',index=False)
manifest.loc[manifest['audit_id'].isin(near_ids)].sort_values('audit_id').to_csv(C/'near_miss_9_blinded_review_manifest_private.csv',index=False)
manifest.loc[manifest['audit_id'].isin(baseline_ids)].sort_values('audit_id').to_csv(C/'baseline16_blinded_review_manifest_private.csv',index=False)
rubric.to_csv(C/'historical_imagery_review_rubric.csv',index=False)
priority.to_csv(C/'near_miss_priority_audit_ids_private.csv',index=False)

# Unblinding key remains private and is not needed during visual review.
key.to_pickle(C/'historical_imagery_unblinding_key_private.pkl')
spatial.to_pickle(C/'historical_imagery_spatial_context_private.pkl')

print('near-miss KML:',near_kml)
print('near-miss review CSV:',C/'near_miss_9_blinded_review_manifest_private.csv')
print('optional baseline reference KML:',baseline_kml)
print('all-25 KML:',all_kml)


## Manual Google Earth Pro review — review the 9 near-miss candidates first

1. Open `near_miss_9_blinded_historical_imagery_review_private.kml` in Google Earth Pro.
2. For each `Ixx candidate`, enable **Historical Imagery**.
3. Start from the candidate's **median observation date** shown in the placemark description.
4. If imagery is unavailable on that exact date, use the nearest interpretable image and record its actual imagery date in the review CSV.
5. Inspect the candidate point and the 50 / 100 / 150 m rings. The HOME marker is only an orientation reference.
6. Fill `near_miss_9_blinded_review_manifest_private.csv` using only the fixed rubric. Do not look at the unblinding key, Stage-07j behavior, or BCL evidence while classifying imagery.

Recommended field meanings:

- `imagery_available`: `yes`, `no`, or `unclear`.
- `imagery_quality`: free text such as `high`, `medium`, `low` (record what you actually see).
- `structure_present`: whether a visible built structure/complex exists at the candidate.
- `visual_context_class`: choose exactly one value from `historical_imagery_review_rubric.csv`.
- `visual_context_confidence`: `high`, `medium`, or `low`.
- `candidate_inside_same_complex`: whether the GPS candidate lies plausibly within the visible complex rather than only near it.
- `historical_name_evidence`: only enter a historical name if independently visible/verified for that period; satellite morphology alone cannot supply it.
- `present_day_name_aid`: optional present-day geocoder/map name; this is a naming aid only, not historical proof.


## Interpretation boundary

- Historical imagery can verify visible structure and broad physical/land-use context, not employment or occupation.
- A commercial/office-like morphology is contextual support, not an OFFICE ground-truth label.
- Residential / vacant / transport morphology can be useful contradictory context, but imagery quality and temporal offset must be recorded.
- Present-day Baidu/Amap/Google labels must never be treated as proof that the same POI/function existed during 2007–2012.
- Complete the blinded review before unblinding against Stage-07j behavior or BCL.
